# 03 — Feature engineering and preprocessing

This notebook turns the cleaned data from 01 into feature matrices ready for modeling in 04. The job it has to get right is the rule Andrew keeps drilling: any step that fits parameters from the data (an imputer's median, an encoder's category list, a scaler's mean/std, a target encoder's per-category rate) is fit on the training set and *then* applied to the test set — never the other way around, never on both pooled together.

Everything in 02's EDA was done on the full sampled dataset. This notebook is where that gets corrected: the time-aware split happens first, then every feature step runs on the train slice only.

By the end there are six matrices and one bundle on disk ready for 04:

- `X_train.csv`, `X_test.csv` — unscaled numeric feature matrices
- `X_train_scaled.csv`, `X_test_scaled.csv` — the standardized versions (for the logistic regression; the tree models don't need them)
- `y_train.csv`, `y_test.csv` — binary targets
- `preprocessors.joblib` — the fitted imputer, one-hot encoder, state target-encoding map, scaler, and the feature-column list. 04 doesn't strictly need this to run, but it means I can re-apply the exact same pipeline to new loans later without rewriting any code.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 60)

## Loading the cleaned data from 01

The CSV has 250k rows, 32 original columns + the binary `target` built in 01 + two parsed datetime columns (`issue_d_dt`, `earliest_cr_line_dt`). Reading with `parse_dates` so the dates come back as timestamps, not strings.

In [2]:
df = pd.read_csv(
    "../data/processed/loan_cleaned.csv",
    parse_dates=["issue_d_dt", "earliest_cr_line_dt"],
)
print(f"shape: {df.shape}")
print(f"default rate: {df['target'].mean()*100:.2f}%")

shape: (250000, 35)
default rate: 21.48%


## Time-aware split — why not just random?

A `train_test_split(..., random_state=42)` would be optimistic on this data. Credit outcomes have temporal structure: interest rates, approval policies and the macroeconomic backdrop all drift year to year. If training rows and test rows come from the same months, the model is implicitly allowed to see the "future" during training — which won't be true at inference time.

The honest alternative is to split by date. Everything issued before some cutoff goes to train, everything after is the holdout. I'm using **2018-01-01** as the cutoff: 2007–2017 for training, 2018 for testing. That mirrors the actual use case — a model trained today has to generalize to loans that haven't been issued yet.

One subtlety I didn't appreciate until I did the split: the "completed loans" filter in 01 drops `Current` loans entirely. So 2018 loans that are still being paid off and haven't gone late aren't in the dataset — only ones that have either been fully paid, charged off or gone late. The test set is biased toward loans with definitive outcomes early in their lifecycle, which almost certainly inflates the test-set default rate relative to a truly representative holdout. 04 will note it as a limitation on any single-number metric.

In [3]:
CUTOFF = pd.Timestamp("2018-01-01")

train_mask = df["issue_d_dt"] < CUTOFF
test_mask = df["issue_d_dt"] >= CUTOFF

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"train: {len(df_train):,} rows, {df_train['target'].mean()*100:.2f}% default, "
      f"{df_train['issue_d_dt'].min():%Y-%m} to {df_train['issue_d_dt'].max():%Y-%m}")
print(f" test: {len(df_test):,} rows, {df_test['target'].mean()*100:.2f}% default, "
      f"{df_test['issue_d_dt'].min():%Y-%m} to {df_test['issue_d_dt'].max():%Y-%m}")

train: 238,061 rows, 21.21% default, 2007-06 to 2017-12
 test: 11,939 rows, 27.05% default, 2018-01 to 2018-12


One thing to notice in the sizes above: the test set is small relative to train (2018 is one year out of eleven in the dataset). It's not nothing, but single-year holdouts have high variance — if 2018 happened to be an unusual year for defaults, the test metric would reflect that rather than a stable property of the model. 04 will mention this when it reports metrics; for a production project I'd use a rolling or expanding window and average across several holdouts.

## Row-wise features — stateless, so I can compute them on both sides the same way

Six features are just transformations of existing columns, with no parameters learned from data. These are safe to compute on train and test identically, with no "fit" step:

- `fico_avg` — midpoint of the FICO low/high range (the two columns are redundant on their own).
- `credit_history_years` — `(issue_d − earliest_cr_line)` in years. Uses only information knowable at origination (no hard-coded reference date).
- `loan_to_income` — `loan_amnt / annual_inc`. A ratio of two existing columns.
- `monthly_debt_burden` — `installment / (annual_inc / 12)`. Share of monthly income going to the loan payment.
- `emp_length_num` — the `<1 year` / `1 year` / … / `10+ years` string mapped to a float, so a linear model can use it as a numeric feature.
- `grade_num` — Lending Club's A–G grade mapped to 7–1 so the ordering carries into the model.

Writing them as a single function so I apply the same transformation to train and test without accidental drift.

In [4]:
def add_rowwise_features(df):
    df = df.copy()

    df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

    df["credit_history_years"] = (
        (df["issue_d_dt"] - df["earliest_cr_line_dt"]).dt.days / 365
    )

    # protect against zero / near-zero income blowing up the ratios
    safe_income = df["annual_inc"].replace(0, np.nan)
    df["loan_to_income"] = df["loan_amnt"] / safe_income
    df["monthly_debt_burden"] = df["installment"] / (safe_income / 12)

    emp_map = {
        "< 1 year": 0.5, "1 year": 1, "2 years": 2, "3 years": 3,
        "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
        "8 years": 8, "9 years": 9, "10+ years": 10,
    }
    df["emp_length_num"] = df["emp_length"].map(emp_map)

    grade_map = {"A": 7, "B": 6, "C": 5, "D": 4, "E": 3, "F": 2, "G": 1}
    df["grade_num"] = df["grade"].map(grade_map)

    return df

df_train = add_rowwise_features(df_train)
df_test = add_rowwise_features(df_test)

new_cols = ["fico_avg", "credit_history_years", "loan_to_income",
            "monthly_debt_burden", "emp_length_num", "grade_num"]
df_train[new_cols].head()

,fico_avg,credit_history_years,loan_to_income,monthly_debt_burden,emp_length_num,grade_num
0,742.0,18.430137,0.083232,0.030959,10.0,7
1,672.0,29.356164,0.269841,0.110152,2.0,5
2,697.0,8.926027,0.140000,0.053106,3.0,7
3,667.0,12.512329,0.035661,0.013212,3.0,7
4,727.0,8.920548,0.200000,0.075097,9.0,7


## Deciding what goes into the feature matrix

Dropping a handful of columns before anything else touches them:

- `loan_status` — literally encodes the outcome. Pure leakage.
- `grade`, `sub_grade`, `emp_length` — the numeric versions (`grade_num`, `emp_length_num`) already capture these; the string versions would duplicate the signal and need their own encoder.
- `fico_range_low`, `fico_range_high` — rolled into `fico_avg`.
- `issue_d`, `earliest_cr_line` (string columns) and their `_dt` siblings — the `_dt` versions were useful for the time-aware split and for computing `credit_history_years`, but as features themselves they'd leak the time of issuance, which the model shouldn't condition on directly.

What's left: `addr_state` (needs encoding), `home_ownership` / `verification_status` / `purpose` (one-hot), and ~22 numeric columns.

In [5]:
drop_cols = [
    "loan_status",
    "grade", "sub_grade", "emp_length",
    "fico_range_low", "fico_range_high",
    "issue_d", "earliest_cr_line", "issue_d_dt", "earliest_cr_line_dt",
]
df_train = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns])
df_test = df_test.drop(columns=[c for c in drop_cols if c in df_test.columns])

print(f"columns remaining: {len(df_train.columns)}")
print(df_train.columns.tolist())

columns remaining: 31
['loan_amnt', 'term', 'int_rate', 'installment', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'target', 'fico_avg', 'credit_history_years', 'loan_to_income', 'monthly_debt_burden', 'emp_length_num', 'grade_num']


In [6]:
y_train = df_train["target"].copy()
y_test = df_test["target"].copy()

X_train = df_train.drop(columns=["target"])
X_test = df_test.drop(columns=["target"])

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print(f"numeric columns:     {len(numeric_cols)}")
print(f"categorical columns: {len(categorical_cols)} — {categorical_cols}")

numeric columns:     26
categorical columns: 4 — ['home_ownership', 'verification_status', 'purpose', 'addr_state']


/var/folders/jt/_cgj1vlj3d14nfyz_nv0sm8w0000gn/T/ipykernel_10186/3387932513.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


## Imputation — the first step that fits parameters on train only

I'm using the median rather than the mean because several of these columns (`revol_bal`, `tot_coll_amt`, `annual_inc`) have heavy right tails that would drag a mean around. Median is more robust.

The specialization's rule shows up concretely here: `fit_transform(X_train)` learns one median per numeric column from train rows only; `transform(X_test)` applies those same medians to test. If I used `fit_transform` on the pooled data, I'd be using test rows to decide what the "typical" value for each column is — which is a form of leakage.

In [7]:
imputer = SimpleImputer(strategy="median")
X_train[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = imputer.transform(X_test[numeric_cols])

assert X_train[numeric_cols].isna().sum().sum() == 0
assert X_test[numeric_cols].isna().sum().sum() == 0
print(f"imputed {len(numeric_cols)} numeric columns; no NaN remaining")

imputed 26 numeric columns; no NaN remaining


## One-hot encoding the small-cardinality nominals

`home_ownership`, `verification_status` and `purpose` all have a manageable number of categories (3–14), so one-hot is the obvious encoding — each category becomes its own 0/1 column. The encoder is fit on train (so it learns the category list from training data only) and then applied to test. `handle_unknown='ignore'` means that if the test set ever contains a category the training data didn't have, the encoder outputs all zeros for that row rather than raising. Unlikely at this scale, but the kind of defensive setting you want in production.

In [8]:
ohe_cols = ["home_ownership", "verification_status", "purpose"]
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe = ohe.transform(X_test[ohe_cols])

ohe_feature_names = ohe.get_feature_names_out(ohe_cols)
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe_feature_names, index=X_train.index)
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe_feature_names, index=X_test.index)

print(f"one-hot produced {len(ohe_feature_names)} columns from {len(ohe_cols)} source columns")

one-hot produced 23 columns from 3 source columns


## State encoding — smoothed target encoding

`addr_state` has 50 values. One-hotting would add 50 nearly-empty columns, most of which carry very little information (the small states just don't have enough loans to be learnable). An alternative that isn't in the course curriculum but is standard in industry when category cardinality is high: **target encoding** — replace each state with its default rate on the training set.

Target encoding has a well-known failure mode. For a rare state with just a few loans in train, the per-state rate is noisy and the encoder essentially memorizes those rows. The fix is **smoothing** — pool each state's local rate toward the global default rate, with the weight on the local rate growing as the sample size grows:

> smoothed_rate(state) = (n / (n + k)) · local_rate + (k / (n + k)) · global_rate

where `n` is the number of training loans in that state, `local_rate` is the state's own default rate, `global_rate` is the overall train default rate and `k` is a smoothing constant. Larger `k` → more pull toward the global mean. I'm using **k = 50**, so a state with exactly 50 training loans gets equal weight on local and global.

Fit on train only. Test states get looked up in the trained map; any state the train set didn't see (unlikely with 50 US states but possible for territories) gets the global mean as a sensible fallback.

In [9]:
K_SMOOTH = 50

global_rate = y_train.mean()

state_stats = (
    pd.DataFrame({"state": X_train["addr_state"].values, "target": y_train.values})
    .groupby("state")
    .agg(n=("target", "size"), rate=("target", "mean"))
)
state_stats["smoothed"] = (
    state_stats["n"] / (state_stats["n"] + K_SMOOTH) * state_stats["rate"]
    + K_SMOOTH / (state_stats["n"] + K_SMOOTH) * global_rate
)
state_encoding = state_stats["smoothed"].to_dict()

X_train["state_enc"] = X_train["addr_state"].map(state_encoding).fillna(global_rate)
X_test["state_enc"] = X_test["addr_state"].map(state_encoding).fillna(global_rate)

print(f"global default rate (train): {global_rate*100:.2f}%")
print(f"\nhighest smoothed rates:")
print(state_stats.sort_values("smoothed", ascending=False).head(3)[["n", "rate", "smoothed"]])
print(f"\nlowest smoothed rates:")
print(state_stats.sort_values("smoothed").head(3)[["n", "rate", "smoothed"]])

global default rate (train): 21.21%

highest smoothed rates:
          n      rate  smoothed
state                          
NE      602  0.305648  0.298470
MS     1180  0.299153  0.295612
AL     2853  0.256222  0.255461

lowest smoothed rates:
          n      rate  smoothed
state                          
OR     2872  0.145891  0.147023
NH     1119  0.146559  0.149361
WV      871  0.151550  0.154834


A sanity check in the output above: the small states with extreme local rates get pulled hard toward the global mean. A state with 100 loans and a local rate of 30% ends up with a smoothed rate around 27% (local gets 2/3 of the weight, global gets 1/3). A state with 10,000 loans at 30% stays essentially at 30% because the smoothing weight is swamped. That's the behavior I want.

## Assembling the final feature matrices

Drop the raw categorical columns (now encoded) and concatenate the one-hot block onto the numeric block. Both matrices get the same column order so 04 can treat them identically.

In [10]:
X_train = X_train.drop(columns=ohe_cols + ["addr_state"])
X_test = X_test.drop(columns=ohe_cols + ["addr_state"])

X_train = pd.concat([X_train, X_train_ohe_df], axis=1)
X_test = pd.concat([X_test, X_test_ohe_df], axis=1)

# defensive: make sure both sides have the same column set in the same order
missing_in_test = set(X_train.columns) - set(X_test.columns)
missing_in_train = set(X_test.columns) - set(X_train.columns)
for col in missing_in_test:
    X_test[col] = 0.0
for col in missing_in_train:
    X_train[col] = 0.0
X_test = X_test[X_train.columns]

print(f"X_train shape: {X_train.shape}")
print(f" X_test shape: {X_test.shape}")

X_train shape: (238061, 50)
 X_test shape: (11939, 50)


## Scaling — fit on train only, mostly for the linear model

Logistic regression needs features on comparable scales. A feature in dollars (`loan_amnt`, up to around 40,000) will dominate a feature in percent (`int_rate`, up to around 30) in the gradient update unless they're normalized — Andrew's week-2 lecture on feature scaling makes this exact point with a contour-plot animation.

Tree ensembles (random forest, XGBoost) don't care about scale — they split on thresholds, which are invariant to any monotone transformation of a feature. So I'll fit a `StandardScaler` on the train features, save both the scaled and unscaled matrices, and let 04 pick which one each model needs.

In [11]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# sanity: scaled train should have mean ~0 and std ~1 per column (approximately)
print(f"scaled train means: min={X_train_scaled.mean().min():.3f}, max={X_train_scaled.mean().max():.3f}")
print(f"scaled train stds:  min={X_train_scaled.std().min():.3f}, max={X_train_scaled.std().max():.3f}")

scaled train means: min=-0.000, max=0.000
scaled train stds:  min=1.000, max=1.000


## Save everything 04 needs

In [12]:
out_dir = Path("../data/processed")

X_train.to_csv(out_dir / "X_train.csv", index=False)
X_test.to_csv(out_dir / "X_test.csv", index=False)
X_train_scaled.to_csv(out_dir / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(out_dir / "X_test_scaled.csv", index=False)
y_train.to_csv(out_dir / "y_train.csv", index=False, header=["target"])
y_test.to_csv(out_dir / "y_test.csv", index=False, header=["target"])

joblib.dump(
    {
        "imputer": imputer,
        "ohe": ohe,
        "ohe_cols": ohe_cols,
        "state_encoding": state_encoding,
        "global_default_rate": global_rate,
        "scaler": scaler,
        "feature_columns": X_train.columns.tolist(),
        "numeric_columns": numeric_cols,
        "k_smooth": K_SMOOTH,
        "cutoff_date": str(CUTOFF.date()),
    },
    out_dir / "preprocessors.joblib",
    compress=3,
)

print(f"saved to {out_dir}:")
for p in sorted(out_dir.glob("*")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:30s}  {size_mb:7.2f} MB")

saved to ../data/processed:
  .gitkeep                           0.00 MB
  X_test.csv                         3.46 MB
  X_test_scaled.csv                 12.10 MB
  X_train.csv                       69.10 MB
  X_train_scaled.csv               241.41 MB
  loan_cleaned.csv                  55.72 MB
  preprocessors.joblib               0.00 MB
  summary_by_grade.csv               0.00 MB
  summary_by_purpose.csv             0.00 MB
  summary_by_state.csv               0.00 MB
  y_test.csv                         0.02 MB
  y_train.csv                        0.48 MB


## What I took away from building this pipeline

**Figuring out what counts as "fits parameters from data" is the actual skill.** Row-wise construction (ratios, date differences, mapping a grade letter to a number) is stateless — I can do it anywhere, no fitting involved. Everything else — imputation, encoding, scaling, target encoding — looks across rows to decide something, and that "looking across rows" is the exact thing that has to be confined to the training set. The rule "fit preprocessing on train only" sounds easy; applying it required explicitly asking, for every single step, whether it looks across rows. A couple of steps I'd have moved to the wrong side if I hadn't.

**Target encoding felt mildly unsettling.** The feature value for a state is literally the training-set default rate for that state, which means I'm using the target itself to build a feature. The smoothing is what keeps it safe-ish on small states — without it, a state with 10 loans and 5 defaults would be encoded as 0.5 and the model would memorize those 10 rows. With `k = 50`, a state with 10 loans gets its local rate pulled 83% of the way back toward the global average. That felt like the right amount of humility for a sample that small.

**Two things to carry into notebook 04.** The test set is one year of data, which will produce noisier metrics than I'd like. And the "completed loans only" filter means the test rows are biased toward loans with definitive outcomes early in their lifecycle; the real-world default rate on 2018-issued loans will be lower than the test-set rate suggests. 04 will mention both when it reports numbers.